In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__)), "em-master"))  # Optional: activates local environment

# Optional: define data and code directory relative to current script
script_dir = dirname(dirname(@__DIR__))                      
em_dir = joinpath(script_dir, "em-master")  # EM algorithm folder
study_file = joinpath(script_dir, "Data", "Processed", "Study2.csv") #change to study1.csv for study1


# Set model and EM algorithm parameters
full = false               # Use diagonal covariance matrix if 'false'
emtol = 1e-3               # EM stopping threshold based on relative change
push!(LOAD_PATH, em_dir)  # Add directory for accessing custom files

# Load required packages
using EM, Distributions, LinearAlgebra, Optim, Statistics, Random
using GLM, DataFrames, NamedArrays, SpecialFunctions, CSV

# Load dataset
data = CSV.read(study_file, DataFrame)

# Model parameters
NS = 57                       # Number of subjects, change to 47 for study1
subs = 1:NS                   # Subject indices
X = ones(Int, NS)             # Placeholder variable for EM (model-specific)
startbetas = [0 0 0 0]
startsigma = [1, 1, 1, 1]     # Initial sigma values

# Model function to calculate negative log-likelihood
function M2(params, data)
    lr_params = @. 1 / (1 + exp(-params[1:3]))
    Conf_True, Conf_False, Disconf_False = lr_params
    
    Disconf_True = Conf_True
    
    # Transform beta for Q-value scaling
    Beta = 25 * (1 + erf(params[4] / sqrt(2)))
    
    # Extract data columns (using multiple assignment)
    R, C1, C2, TrialType, FeedbackType, Condition = (
        data.Reward, data.C1, data.C2, data.TrialType, 
        data.Feedback, data.Condition
    )
    
    # Initialize Q-values matrix
    Q = zeros(typeof(Beta), 2, 4)
    lik = 0.0
    
    for t in eachindex(C1)
        QC = Condition[t]
        chosen = C1[t] == 1 ? 1 : 2
        unchosen = 3 - chosen
        
        # Update likelihood based on choice probability
        lik += -log1p(exp(-Beta * (Q[chosen, QC] - Q[unchosen, QC])))
        
        # Calculate prediction error
        is_chosen_feedback = FeedbackType[t] == 1
        pe = is_chosen_feedback ? 
             (R[t] - Q[chosen, QC]) : 
             (R[t] - Q[unchosen, QC])
        
        # Determine learning rates based on trial type and prediction error
        is_true_trial = TrialType[t] == 1
        is_positive_pe = pe > 0
        
        # Select appropriate learning rate based on conditions
        function get_learning_rate(is_factual)
            if is_true_trial
                if is_factual == is_positive_pe
                    return Conf_True
                else
                    return Disconf_True
                end
            else
                if is_factual == is_positive_pe
                    return Conf_False
                else
                    return Disconf_False
                end
            end
        end
        
        # Update Q-values separately for chosen and unchosen options
        options = [(1, chosen), (2, unchosen)]  # Pairs of (index, option)
        
        for (option_index, current_option) in options
            # For chosen feedback (1): update chosen option (index 1)
            # For unchosen feedback (0): update unchosen option (index 2)
            is_correct_option_to_update = (option_index == 1 && is_chosen_feedback) || 
                                        (option_index == 2 && !is_chosen_feedback)
            
            if is_correct_option_to_update
                lr = get_learning_rate(is_chosen_feedback)
                Q[current_option, QC] = (1 - lr) * Q[current_option, QC] + lr * R[t]
            end
        end
    end
    
    return -lik
end
# Run the EM optimization on the model with initial parameters
(betas, sigma, x, l, h) = em(data, subs, X, startbetas, startsigma, M2, maxiter=2000, emtol=emtol, full=full)


In [ ]:
using CSV
using DataFrames
#compute unbiased per subject marginal likelihoods via cross validation.

liks = loocv(data,subs,x,X,betas,sigma,M2;maxiter=2000,emtol=emtol, full=full)
sum(liks)

# Convert the Vector to a DataFrame
liks_df = DataFrame(Values = liks)

CSV.write("M2LOOCV.csv", liks_df)